In [ ]:
import grain
import jax
from context_flux_no.data import TheWellDataSource


jax.config.update("jax_default_device", jax.devices("gpu")[3])

In [ ]:
jax.devices()

In [ ]:
source = TheWellDataSource(
    "../../data/datasets",
    "euler_multi_quadrants_periodicBC",
    include_filters=["gamma_1.3_CO2"],
    window_size=11,
    downsample_spatial=8,
    exclude_field_names=["pressure"],
)
sampler = grain.samplers.IndexSampler(len(source), shuffle=True, seed=0)
transforms = [grain.transforms.Batch(batch_size=64)]

In [ ]:
dataloader = grain.DataLoader(
    data_source=source,
    sampler=sampler,
    operations=transforms,
    worker_count=6,
    # read_options=grain.ReadOptions(num_threads=32, prefetch_buffer_size=1000),
    shard_options=grain.sharding.ShardByJaxProcess(),
)

In [ ]:
dataiter = iter(dataloader)
batch = next(dataiter)

In [ ]:
type(batch)

In [ ]:
batch

In [ ]:
ds = grain.MapDataset.source(source).seed(seed=0).shuffle().batch(batch_size=64)
# it = iter(ds)

# for _ in range(1):
#     next(it)

In [ ]:
performance_config = grain.experimental.pick_performance_config(
    ds=ds, ram_budget_mb=10000, max_workers=8, max_buffer_size=None
)

In [ ]:
performance_config